# 04 — Feature Engineering

**Role:** Member 2 — Feature Engineering  
**Branch:** `member2-features`

This notebook uses the standardized dataset interface produced by Member 1.

It does **not** invent dataset columns. Before running the feature extraction cells,
set the standardized input path and explicitly provide the text/context column names
from Member 1's output.

### Outputs
- TF-IDF text features
- Lightweight text statistics
- Encoded contextual features
- Row-level context-availability mask
- Context-availability fraction

Raw datasets remain outside Git because `data/raw/` and `data/processed/` are gitignored.


In [ ]:
from pathlib import Path
import sys
import pandas as pd

# Make the repository's src/ directory importable from this notebook.
REPO_ROOT = Path.cwd().resolve()
if not (REPO_ROOT / "src").exists():
    REPO_ROOT = REPO_ROOT.parent

sys.path.insert(0, str(REPO_ROOT))

from src.features.text_features import fit_text_features
from src.features.context_features import fit_context_features


## 1. Configure the standardized input

The repository ZIP currently contains no processed dataset, so the path below
must be updated after Member 1 creates the standardized output.

Do not guess the column names. Copy them from Member 1's standardized interface.


In [ ]:
# Update these values after Member 1's standardized dataset is available.
DATA_PATH = REPO_ROOT / "data" / "processed" / "STANDARDIZED_INPUT.csv"

# Required: exact standardized text column name.
TEXT_COLUMN = None

# Optional: exact standardized contextual columns.
# Split them by type according to the actual data dictionary/interface.
NUMERIC_CONTEXT_COLUMNS = []
CATEGORICAL_CONTEXT_COLUMNS = []

# Example supported input extensions:
# .csv -> pd.read_csv
# .parquet -> pd.read_parquet


In [ ]:
def load_standardized_data(path: Path) -> pd.DataFrame:
    if not path.exists():
        raise FileNotFoundError(
            f"Standardized input not found at: {path}. "
            "Wait for Member 1's standardized output, then update DATA_PATH."
        )

    suffix = path.suffix.lower()
    if suffix == ".csv":
        return pd.read_csv(path)
    if suffix == ".parquet":
        return pd.read_parquet(path)

    raise ValueError(
        f"Unsupported file type: {suffix}. Use a CSV or Parquet standardized file."
    )


df = load_standardized_data(DATA_PATH)
print(f"Rows: {len(df):,}")
print(f"Columns: {len(df.columns)}")
print(list(df.columns))


In [ ]:
# Validate the interface explicitly before feature extraction.
if TEXT_COLUMN is None:
    raise ValueError(
        "TEXT_COLUMN is not configured. Use the exact standardized text column from Member 1."
    )

if TEXT_COLUMN not in df.columns:
    raise KeyError(f"TEXT_COLUMN '{TEXT_COLUMN}' is not present in the standardized dataset.")

all_context_columns = NUMERIC_CONTEXT_COLUMNS + CATEGORICAL_CONTEXT_COLUMNS
missing_context = [c for c in all_context_columns if c not in df.columns]
if missing_context:
    raise KeyError(
        f"These configured context columns are not present in the standardized dataset: "
        f"{missing_context}"
    )

print("Text column:", TEXT_COLUMN)
print("Numeric context columns:", NUMERIC_CONTEXT_COLUMNS)
print("Categorical context columns:", CATEGORICAL_CONTEXT_COLUMNS)


## 2. Inspect missingness

This is an inspection step only. No values or columns are fabricated.


In [ ]:
selected_columns = [TEXT_COLUMN] + all_context_columns
display(df[selected_columns].isna().sum().sort_values(ascending=False))


## 3. Build text features

TF-IDF should ultimately be **fit on the training split only** in the real experiment.
This cell is the feature-extraction demonstration; later model notebooks should reuse
the fitted vectorizer for validation/test/cross-dataset data.


In [ ]:
text_result = fit_text_features(
    df,
    TEXT_COLUMN,
    max_features=5000,
    ngram_range=(1, 2),
    min_df=1,
)

print("TF-IDF matrix shape:", text_result.tfidf_matrix.shape)
print("Number of TF-IDF terms:", len(text_result.feature_names))

display(text_result.statistics.head())


## 4. Build contextual features and context-availability mask

The availability mask is `1` when at least one explicitly configured context field is
present for a row, and `0` when all configured context fields are missing.


In [ ]:
context_result = fit_context_features(
    df,
    numeric_columns=NUMERIC_CONTEXT_COLUMNS,
    categorical_columns=CATEGORICAL_CONTEXT_COLUMNS,
)

print("Context matrix shape:", context_result.matrix.shape)
print("Context feature count:", len(context_result.feature_names))
print(
    "Rows with context:",
    int(context_result.availability_mask.sum()),
    "/",
    len(context_result.availability_mask),
)

display(
    pd.DataFrame(
        {
            "context_available": context_result.availability_mask,
            "context_availability_fraction": context_result.availability_fraction,
        }
    ).head()
)


## 5. Save derived feature metadata

The derived feature matrices themselves may be saved locally for downstream notebooks.
Do not add raw datasets or large generated model artifacts to Git.


In [ ]:
FEATURE_DIR = REPO_ROOT / "data" / "processed"
FEATURE_DIR.mkdir(parents=True, exist_ok=True)

# Save small, human-readable feature metadata only.
metadata = pd.DataFrame(
    {
        "text_feature_count": [len(text_result.feature_names)],
        "context_feature_count": [len(context_result.feature_names)],
        "context_rows_available": [int(context_result.availability_mask.sum())],
        "total_rows": [len(df)],
    }
)

metadata_path = FEATURE_DIR / "feature_metadata.csv"
metadata.to_csv(metadata_path, index=False)

print("Saved:", metadata_path)


## Interface handoff to later members

Member 3 and Member 4 should consume the standardized outputs rather than rewriting
preprocessing:

- `text_result.tfidf_matrix`
- `text_result.statistics`
- `context_result.matrix`
- `context_result.availability_mask`
- `context_result.availability_fraction`

Before integration, the team should agree on the exact serialized format for these
matrices and the final train/validation/test split handling.
